# qwen25-arc-kfp-rag

[![Open in JupyterLab](https://img.shields.io/badge/Open%20in-JupyterLab-F37626?logo=jupyter&logoColor=white)](http://localhost:8888/lab/tree/git-miramar-labs-org/projects/qwen25-arc-kfp-rag/notebook.ipynb)

| | |
|---|---|
| **Type** | KFP v2 RAG pipeline with eval-first design |
| **Host** | dgx |

RAG pipeline for general QA using qwen25-arc (Qwen2.5-7B + ARC LoRA) on DGX Spark via KFP

**Development workflow:**
1. Edit `config.yaml` — set `llm.base_url`, `llm.model`, eval thresholds
2. Add domain documents to `docs_src/` (`.txt` or `.md`)
3. Edit `eval_dataset.jsonl` — replace stub rows with real Q&A pairs
4. Fill in each `# ---- USER CODE BLOCK ----` below (see `WORKBOOK.md` for guidance)
5. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
6. Trigger **Deploy to KFP** (or run **Compile & Submit** below)

**Pipeline DAG:**
```
ingest_documents
  → retrieval_eval
      → generation_eval
          → faithfulness_eval ─┐
          → safety_eval ───────┤
                               └→ deployment_gate
```

All steps run CPU-only. `ingest_documents` and `deployment_gate` are fully implemented.
Implement the four eval steps in order: retrieval_eval → generation_eval → faithfulness_eval → safety_eval.

## Step development

- **All imports must be inside the function body** — each component runs in its own container
- Use `Input[Artifact]` / `Output[Artifact]` to pass data between steps
- CPU-only steps: do NOT call `.set_accelerator_type(...)` in the pipeline cell
- LLM endpoint (`llm_base_url`) must be an active serving project; judge endpoint (`judge_base_url`) is usually local Ollama

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Artifact

### ingest_documents

Reads all `.txt` and `.md` files from the PVC input directory (copied from `docs_src/` by
`scripts/deploy_pipeline.py`), chunks them with `RecursiveCharacterTextSplitter`, embeds with
`BAAI/bge-small-en-v1.5` on CPU, and upserts to Qdrant. Recreates the collection on every run.

**Fully implemented — no changes needed.**

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "sentence-transformers>=3.0",
        "qdrant-client>=1.9",
        "langchain-text-splitters>=0.3",
        "pyyaml",
        "mlflow",
    ],
)
def ingest_documents(
    run_id: str,
    qdrant_url: str,
    collection: str,
    embedding_model: str,
    chunk_size: int,
    chunk_overlap: int,
    batch_size: int,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    project_name: str = "qwen25-arc-kfp-rag",
):
    """Chunk docs, embed with sentence-transformers, upsert to Qdrant."""
    import pathlib, mlflow
    from sentence_transformers import SentenceTransformer
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    docs_dir = pathlib.Path("/root/.cache/huggingface/rag-input") / project_name / "docs"
    if not docs_dir.exists():
        raise FileNotFoundError(
            f"Documents directory not found: {docs_dir}\n"
            "Run scripts/deploy_pipeline.py to copy docs_src/ to the PVC first."
        )

    all_files = sorted(docs_dir.glob("**/*.txt")) + sorted(docs_dir.glob("**/*.md"))
    if not all_files:
        raise ValueError(f"No .txt or .md files found in {docs_dir}")

    texts, doc_ids = [], []
    for path in all_files:
        texts.append(path.read_text(encoding="utf-8", errors="ignore"))
        doc_ids.append(path.stem)
    print(f"Loaded {len(texts)} documents from {docs_dir}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks, chunk_metas = [], []
    for text, doc_id in zip(texts, doc_ids):
        for i, chunk in enumerate(splitter.split_text(text)):
            chunks.append(chunk)
            chunk_metas.append({"doc_id": doc_id, "chunk_index": i, "run_id": run_id})
    print(f"Chunked into {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")

    model = SentenceTransformer(embedding_model, device="cpu")
    embeddings = model.encode(chunks, batch_size=batch_size, show_progress_bar=True)
    dim = embeddings.shape[1]
    print(f"Embedded {len(embeddings)} chunks with {embedding_model} (dim={dim})")

    client = QdrantClient(url=qdrant_url)
    try:
        client.delete_collection(collection_name=collection)
    except Exception:
        pass
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
    points = [
        PointStruct(id=i, vector=emb.tolist(), payload={"text": chunk, **meta})
        for i, (chunk, emb, meta) in enumerate(zip(chunks, embeddings, chunk_metas))
    ]
    for i in range(0, len(points), batch_size):
        client.upsert(collection_name=collection, points=points[i : i + batch_size])
    print(f"Upserted {len(points)} chunks to Qdrant collection '{collection}' at {qdrant_url}")

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-ingest"):
        mlflow.log_param("n_documents", len(texts))
        mlflow.log_param("n_chunks", len(chunks))
        mlflow.log_param("embedding_model", embedding_model)
        mlflow.log_param("chunk_size", chunk_size)
        mlflow.log_param("chunk_overlap", chunk_overlap)
        mlflow.log_param("collection", collection)

### retrieval_eval

Evaluates Qdrant retrieval quality: embed each eval question, search top-k,
check if `gold_doc_ids` from `eval_dataset.jsonl` appear in the results.

**Fill in the `USER CODE BLOCK` below.** See `WORKBOOK.md` Step 1 for the full pattern.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "sentence-transformers>=3.0",
        "qdrant-client>=1.9",
        "mlflow",
    ],
)
def retrieval_eval(
    run_id: str,
    qdrant_url: str,
    collection: str,
    embedding_model: str,
    top_k: int,
    eval_sample_size: int,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    retrieval_metrics: Output[Artifact],
    project_name: str = "qwen25-arc-kfp-rag",
):
    """Evaluate retrieval quality: recall@k, MRR, hit rate against eval_dataset.jsonl."""
    import json, pathlib, mlflow
    from sentence_transformers import SentenceTransformer
    from qdrant_client import QdrantClient

    eval_path = (
        pathlib.Path("/root/.cache/huggingface/rag-input") / project_name / "eval_dataset.jsonl"
    )
    if not eval_path.exists():
        raise FileNotFoundError(f"Eval dataset not found: {eval_path}")
    eval_rows = [
        json.loads(l) for l in eval_path.read_text().splitlines() if l.strip()
    ][:eval_sample_size]
    print(f"Loaded {len(eval_rows)} eval rows")

    model = SentenceTransformer(embedding_model, device="cpu")
    client = QdrantClient(url=qdrant_url)

    # ---- USER CODE BLOCK: retrieval loop ----
    # Embed each question, search Qdrant, compute recall@1, recall@5, MRR, hit_rate.
    # Rows with empty gold_doc_ids are skipped for precision/recall (but still counted for hit_rate).
    # See WORKBOOK.md Step 1 for the full implementation pattern.
    recall_at_1, recall_at_5, mrr, hit_rate = 0.0, 0.0, 0.0, 0.0
    # ---- END USER CODE BLOCK ----

    metrics = {
        "recall_at_1": recall_at_1,
        "recall_at_5": recall_at_5,
        "mrr": mrr,
        "hit_rate": hit_rate,
        "n_eval_rows": len(eval_rows),
    }
    pathlib.Path(retrieval_metrics.path).write_text(json.dumps(metrics, indent=2))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-retrieval-eval"):
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
    print(f"Retrieval eval: recall@5={recall_at_5:.3f}, mrr={mrr:.3f}")

### generation_eval

Full RAG chain: retrieve top-k chunks → build prompt → call LLM → LLM-as-judge for correctness.
Writes `generation_results.jsonl` for downstream faithfulness and safety eval.

**Fill in the `USER CODE BLOCK` below.** See `WORKBOOK.md` Step 2 for the full pattern.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "sentence-transformers>=3.0",
        "qdrant-client>=1.9",
        "openai",
        "mlflow",
    ],
)
def generation_eval(
    run_id: str,
    qdrant_url: str,
    collection: str,
    embedding_model: str,
    top_k: int,
    llm_base_url: str,
    llm_model: str,
    max_new_tokens: int,
    eval_sample_size: int,
    judge_model: str,
    judge_base_url: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    generation_results: Output[Artifact],
    generation_metrics: Output[Artifact],
    project_name: str = "qwen25-arc-kfp-rag",
):
    """Retrieve + generate answers via RAG chain. LLM-as-judge for correctness and fact coverage."""
    import json, pathlib, mlflow
    from sentence_transformers import SentenceTransformer
    from qdrant_client import QdrantClient
    from openai import OpenAI

    eval_path = (
        pathlib.Path("/root/.cache/huggingface/rag-input") / project_name / "eval_dataset.jsonl"
    )
    eval_rows = [
        json.loads(l) for l in eval_path.read_text().splitlines() if l.strip()
    ][:eval_sample_size]

    embed_model = SentenceTransformer(embedding_model, device="cpu")
    qdrant = QdrantClient(url=qdrant_url)
    llm_client = OpenAI(base_url=llm_base_url, api_key="not-used")
    judge_client = OpenAI(base_url=judge_base_url, api_key="ollama")

    # ---- USER CODE BLOCK: RAG chain ----
    # For each eval row:
    # 1. Embed the question, retrieve top-k chunks from Qdrant
    # 2. Build a RAG prompt: system context + retrieved chunks + question
    # 3. Call llm_client to generate an answer
    # 4. Call judge_client to score: answer_correctness (1-5), fact_coverage (0-1)
    # 5. Append result dict to `results`
    # See WORKBOOK.md Step 2 for the full implementation pattern.
    results = []
    # ---- END USER CODE BLOCK ----

    pathlib.Path(generation_results.path).write_text(
        "\n".join(json.dumps(r) for r in results) + "\n"
    )
    avg_correctness = sum(r.get("answer_correctness", 0) for r in results) / max(len(results), 1)
    avg_fact_coverage = sum(r.get("fact_coverage", 0.0) for r in results) / max(len(results), 1)
    gen_metrics = {
        "avg_answer_correctness": avg_correctness,
        "avg_fact_coverage": avg_fact_coverage,
        "n_generated": len(results),
    }
    pathlib.Path(generation_metrics.path).write_text(json.dumps(gen_metrics, indent=2))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-generation-eval"):
        for k, v in gen_metrics.items():
            mlflow.log_metric(k, v)
    print(
        f"Generation eval: avg_correctness={avg_correctness:.3f}, "
        f"avg_fact_coverage={avg_fact_coverage:.3f}"
    )

### faithfulness_eval

LLM-as-judge: checks whether each generated answer is fully grounded in the retrieved context.
Scores faithfulness (1–5), citation coverage (0–1), and unsupported claim rate.

**Fill in the `USER CODE BLOCK` below.** See `WORKBOOK.md` Step 3 for the full pattern.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["openai", "mlflow"],
)
def faithfulness_eval(
    generation_results: Input[Artifact],
    run_id: str,
    judge_model: str,
    judge_base_url: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    faithfulness_metrics: Output[Artifact],
):
    """LLM-as-judge: faithfulness + citation coverage of generated answers."""
    import json, pathlib, mlflow
    from openai import OpenAI

    results = [
        json.loads(l)
        for l in pathlib.Path(generation_results.path).read_text().splitlines()
        if l.strip()
    ]
    client = OpenAI(base_url=judge_base_url, api_key="ollama")

    # ---- USER CODE BLOCK: faithfulness judge ----
    # For each result (question, answer, cited_docs):
    # Ask the judge: is the answer fully supported by the cited docs?
    # Judge should return JSON:
    #   {"faithfulness": 1-5, "citation_coverage": 0.0-1.0,
    #    "unsupported_claims": ["..."], "verdict": "PASS" or "FAIL"}
    # See WORKBOOK.md Step 3 for the full implementation pattern.
    scores = []
    # ---- END USER CODE BLOCK ----

    avg_faithfulness = sum(s.get("faithfulness", 0) for s in scores) / max(len(scores), 1)
    avg_citation_coverage = sum(s.get("citation_coverage", 0.0) for s in scores) / max(len(scores), 1)
    unsupported_rate = sum(1 for s in scores if s.get("verdict") == "FAIL") / max(len(scores), 1)
    metrics = {
        "avg_faithfulness": avg_faithfulness,
        "avg_citation_coverage": avg_citation_coverage,
        "unsupported_claim_rate": unsupported_rate,
        "n_judged": len(scores),
    }
    pathlib.Path(faithfulness_metrics.path).write_text(json.dumps(metrics, indent=2))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-faithfulness-eval"):
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
    print(
        f"Faithfulness eval: avg={avg_faithfulness:.3f}, "
        f"citation_cov={avg_citation_coverage:.3f}, "
        f"unsupported_rate={unsupported_rate:.3f}"
    )

### safety_eval

LLM-as-judge safety evaluation of generated answers. Scores each answer 1–5 and
computes the unsafe response rate.

**Fill in the `USER CODE BLOCK` below.** See `WORKBOOK.md` Step 4 for the full pattern.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["openai", "mlflow"],
)
def safety_eval(
    generation_results: Input[Artifact],
    run_id: str,
    judge_model: str,
    judge_base_url: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    safety_metrics: Output[Artifact],
):
    """LLM-as-judge safety evaluation of generated answers."""
    import json, pathlib, mlflow
    from openai import OpenAI

    results = [
        json.loads(l)
        for l in pathlib.Path(generation_results.path).read_text().splitlines()
        if l.strip()
    ]
    client = OpenAI(base_url=judge_base_url, api_key="ollama")

    # ---- USER CODE BLOCK: safety judge ----
    # For each generated answer, score safety on a 1-5 scale.
    # Judge should return JSON:
    #   {"safety": "PASS" or "FAIL", "score": 1-5, "comment": "brief"}
    # Append float scores to `scores`.
    # See WORKBOOK.md Step 4 for the full implementation pattern.
    scores = []  # list of float scores
    # ---- END USER CODE BLOCK ----

    avg_safety = sum(scores) / max(len(scores), 1)
    unsafe_rate = sum(1 for s in scores if s < 3.5) / max(len(scores), 1)
    metrics = {
        "avg_safety_score": avg_safety,
        "unsafe_response_rate": unsafe_rate,
        "n_judged": len(scores),
    }
    pathlib.Path(safety_metrics.path).write_text(json.dumps(metrics, indent=2))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
    print(f"Safety eval: avg_score={avg_safety:.3f}, unsafe_rate={unsafe_rate:.3f}")

### deployment_gate

Reads all metric artifacts, compares against config thresholds, writes `gate_result.json`
to the PVC, and fails the pipeline if any threshold is not met.

**Fully implemented — no changes needed.**

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["mlflow"],
)
def deployment_gate(
    retrieval_metrics: Input[Artifact],
    generation_metrics: Input[Artifact],
    faithfulness_metrics: Input[Artifact],
    safety_metrics: Input[Artifact],
    run_id: str,
    collection: str,
    min_faithfulness_score: float,
    min_relevancy_score: float,
    min_citation_coverage: float,
    max_unsupported_claim_rate: float,
    min_safety_score: float,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    pipeline_name: str = "qwen25-arc-kfp-rag",
):
    """Gate deployment on retrieval, faithfulness, generation, and safety thresholds."""
    import datetime, json, pathlib, mlflow

    retrieval = json.loads(pathlib.Path(retrieval_metrics.path).read_text())
    generation = json.loads(pathlib.Path(generation_metrics.path).read_text())
    faithfulness = json.loads(pathlib.Path(faithfulness_metrics.path).read_text())
    safety = json.loads(pathlib.Path(safety_metrics.path).read_text())

    recall_at_5      = retrieval.get("recall_at_5", 0.0)
    avg_correctness  = generation.get("avg_answer_correctness", 0.0)
    avg_faithfulness = faithfulness.get("avg_faithfulness", 0.0)
    citation_cov     = faithfulness.get("avg_citation_coverage", 0.0)
    unsupported_rate = faithfulness.get("unsupported_claim_rate", 1.0)
    avg_safety       = safety.get("avg_safety_score", 0.0)

    checks = {
        "retrieval_recall_at_5":  (recall_at_5,       0.70,                      ">="),
        "faithfulness":           (avg_faithfulness,  min_faithfulness_score,     ">="),
        "relevancy":              (avg_correctness,   min_relevancy_score,        ">="),
        "citation_coverage":      (citation_cov,      min_citation_coverage,      ">="),
        "unsupported_claim_rate": (unsupported_rate,  max_unsupported_claim_rate, "<="),
        "safety_score":           (avg_safety,        min_safety_score,           ">="),
    }

    passed = True
    for name, (value, threshold, op) in checks.items():
        ok = value >= threshold if op == ">=" else value <= threshold
        status = "PASS" if ok else "FAIL"
        print(f"  {name}: {value:.4f} {op} {threshold:.4f}  → {status}")
        if not ok:
            passed = False

    gate_status = "PASS" if passed else "FAIL"
    print(f"\nDeployment gate: {gate_status}")

    result = {
        "project": pipeline_name,
        "run_id": run_id,
        "collection": collection,
        "gate": gate_status,
        "metrics": {
            "recall_at_5": round(recall_at_5, 6),
            "avg_faithfulness": round(avg_faithfulness, 6),
            "avg_answer_correctness": round(avg_correctness, 6),
            "citation_coverage": round(citation_cov, 6),
            "unsupported_claim_rate": round(unsupported_rate, 6),
            "avg_safety_score": round(avg_safety, 6),
        },
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }
    out_dir = pathlib.Path(f"/root/.cache/huggingface/rag-runs/{pipeline_name}/{run_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "gate_result.json").write_text(json.dumps(result, indent=2))
    print(f"Gate result written: {out_dir}/gate_result.json")

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-gate"):
        mlflow.log_metric("gate_pass", int(passed))
        for name, (value, _, _op) in checks.items():
            mlflow.log_metric(name, value)

    if not passed:
        raise RuntimeError(
            f"Deployment gate FAILED for {pipeline_name} run {run_id}"
        )

### Pipeline

Wires all steps together. Defaults are read from `config.yaml` at build time.
All steps mount `hf-model-cache` PVC at `/root/.cache/huggingface` and receive
API keys from the `mlabs-api-keys` K8s secret.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_pipeline_name = "qwen25-arc-kfp-rag"

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="qwen25-arc-kfp-rag")
def pipeline(
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "qwen25-arc-kfp-rag",
    qdrant_url: str = _cfg["qdrant"]["url"],
    collection: str = _cfg["qdrant"]["collection"],
    embedding_model: str = _cfg["embedding"]["model"],
    chunk_size: int = _cfg["embedding"]["chunk_size"],
    chunk_overlap: int = _cfg["embedding"]["chunk_overlap"],
    batch_size: int = _cfg["embedding"]["batch_size"],
    llm_base_url: str = _cfg["llm"]["base_url"],
    llm_model: str = _cfg["llm"]["model"],
    top_k: int = _cfg["llm"]["top_k"],
    max_new_tokens: int = _cfg["llm"]["max_new_tokens"],
    eval_sample_size: int = _cfg["eval"]["sample_size"],
    min_faithfulness_score: float = _cfg["eval"]["min_faithfulness_score"],
    min_relevancy_score: float = _cfg["eval"]["min_relevancy_score"],
    min_citation_coverage: float = _cfg["eval"]["min_citation_coverage"],
    max_unsupported_claim_rate: float = _cfg["eval"]["max_unsupported_claim_rate"],
    min_safety_score: float = _cfg["eval"]["min_safety_score"],
    judge_model: str = _cfg["judge"]["model"],
    judge_base_url: str = _cfg["judge"]["base_url"],
):
    ingest = ingest_documents(
        run_id=run_id,
        qdrant_url=qdrant_url,
        collection=collection,
        embedding_model=embedding_model,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        batch_size=batch_size,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        project_name=_pipeline_name,
    )

    ret_eval = retrieval_eval(
        run_id=run_id,
        qdrant_url=qdrant_url,
        collection=collection,
        embedding_model=embedding_model,
        top_k=top_k,
        eval_sample_size=eval_sample_size,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        project_name=_pipeline_name,
    )
    ret_eval.after(ingest)

    gen_eval = generation_eval(
        run_id=run_id,
        qdrant_url=qdrant_url,
        collection=collection,
        embedding_model=embedding_model,
        top_k=top_k,
        llm_base_url=llm_base_url,
        llm_model=llm_model,
        max_new_tokens=max_new_tokens,
        eval_sample_size=eval_sample_size,
        judge_model=judge_model,
        judge_base_url=judge_base_url,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        project_name=_pipeline_name,
    )
    gen_eval.after(ret_eval)

    faith_eval = faithfulness_eval(
        generation_results=gen_eval.outputs["generation_results"],
        run_id=run_id,
        judge_model=judge_model,
        judge_base_url=judge_base_url,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )

    saf_eval = safety_eval(
        generation_results=gen_eval.outputs["generation_results"],
        run_id=run_id,
        judge_model=judge_model,
        judge_base_url=judge_base_url,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )

    gate = deployment_gate(
        retrieval_metrics=ret_eval.outputs["retrieval_metrics"],
        generation_metrics=gen_eval.outputs["generation_metrics"],
        faithfulness_metrics=faith_eval.outputs["faithfulness_metrics"],
        safety_metrics=saf_eval.outputs["safety_metrics"],
        run_id=run_id,
        collection=collection,
        min_faithfulness_score=min_faithfulness_score,
        min_relevancy_score=min_relevancy_score,
        min_citation_coverage=min_citation_coverage,
        max_unsupported_claim_rate=max_unsupported_claim_rate,
        min_safety_score=min_safety_score,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        pipeline_name=_pipeline_name,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = {
        "OPENAI_API_KEY": "OPENAI_API_KEY",
        "HF_TOKEN": "HF_TOKEN",
        "LANGCHAIN_API_KEY": "LANGCHAIN_API_KEY",
    }
    _PVC = "hf-model-cache"
    for _task in [ingest, ret_eval, gen_eval, faith_eval, saf_eval, gate]:
        k8s_ext.mount_pvc(_task, pvc_name=_PVC, mount_path="/root/.cache/huggingface")
        k8s_ext.use_secret_as_env(_task, _SECRET, _SECRET_KEYS)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell to regenerate `pipeline.py`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)